# PF4 Variants from NCBI dbSNP

This notebook extracts SNP variants associated with the PF4 gene from NCBI dbSNP XML data. It parses variant-level information (e.g., rsID, alleles, genomic position, functional annotations) and computes allele frequency metrics across multiple datasets (ALFA, ALSPAC, TOMMO).

In [1]:
import xml.etree.ElementTree as ET
import pandas as pd
import re
import json
from pathlib import Path

In [2]:
xml_path = Path("../data/raw/ncbi/snp_result.xml")
region_path = Path("../data/interim/ensembl/region.json")

out_dir = Path("../data/final")
out_dir.mkdir(parents=True, exist_ok=True)

out_csv = out_dir / "pf4_variants.csv"

In [3]:
def parse_freq(freq_str: str):
    if not freq_str or "=" not in freq_str or "/" not in freq_str:
        return None, None, None

    allele, rest = freq_str.split("=", 1)
    freq_part, ac_part = rest.split("/", 1)

    allele = allele.strip()

    try:
        freq = float(freq_part)
    except ValueError:
        freq = None

    try:
        ac = int(ac_part)
    except ValueError:
        m = re.search(r"\d+", ac_part)
        ac = int(m.group(0)) if m else None

    return allele, freq, ac


def alleles_ref_alt_from_spdi(spdi_text: str):
    if not spdi_text:
        return None

    changes = []
    for sp in [s.strip() for s in spdi_text.split(",") if s.strip()]:
        parts = sp.split(":")
        if len(parts) >= 4:
            ref = parts[-2]
            alt = parts[-1]
            changes.append(f"{ref}>{alt}")

    seen = set()
    out = []
    for c in changes:
        if c not in seen:
            seen.add(c)
            out.append(c)

    return ";".join(out) if out else None

In [4]:
def parse_document_summary(doc):
    snp_id = doc.findtext("SNP_ID")
    rsid = f"rs{snp_id.strip()}" if snp_id else None

    variant_type = doc.findtext("SNP_CLASS")

    chrpos_grch38 = doc.findtext("CHRPOS")
    chrpos_grch37 = doc.findtext("CHRPOS_PREV_ASSM")

    spdi = doc.findtext("SPDI")
    alleles_ref_alt = alleles_ref_alt_from_spdi(spdi)

    gene_names = []
    genes = doc.find("GENES")
    if genes is not None:
        for gene_e in genes.findall("GENE_E"):
            name = gene_e.findtext("NAME")
            if name:
                gene_names.append(name)
    gene_name = ";".join(sorted(set(gene_names))) if gene_names else None

    functional_consequence = doc.findtext("FXN_CLASS")
    validation_flags = doc.findtext("VALIDATED")

    alfa_freq = alfa_ac = None
    alspac_freq = alspac_ac = None
    tommo_freq = tommo_ac = None

    global_mafs = doc.find("GLOBAL_MAFS")
    if global_mafs is not None:
        for maf in global_mafs.findall("MAF"):
            study = maf.findtext("STUDY")
            freq_str = maf.findtext("FREQ")

            _, freq, ac = parse_freq(freq_str)

            if study == "ALFA":
                alfa_freq, alfa_ac = freq, ac
            elif study == "ALSPAC":
                alspac_freq, alspac_ac = freq, ac
            elif study == "TOMMO":
                tommo_freq, tommo_ac = freq, ac

    return {
        "rsID": rsid,
        "Variant_type": variant_type,
        "Alleles_REF_ALT": alleles_ref_alt,
        "CHRPOS_GRCh38": chrpos_grch38,
        "CHRPOS_GRCh37": chrpos_grch37,
        "Canonical_SPDI": spdi,
        "Gene_name": gene_name,
        "Functional_consequence": functional_consequence,
        "Validation_flags": validation_flags,
        "ALFA_freq": alfa_freq,
        "ALFA_AC": alfa_ac,
        "ALSPAC_freq": alspac_freq,
        "ALSPAC_AC": alspac_ac,
        "TOMMO_freq": tommo_freq,
        "TOMMO_AC": tommo_ac,
    }

In [5]:
tree = ET.parse(xml_path)
root = tree.getroot()

docs = root.findall("DocumentSummary")
rows = [parse_document_summary(d) for d in docs]

df = pd.DataFrame(rows)

df.head()

,rsID,Variant_type,Alleles_REF_ALT,CHRPOS_GRCh38,CHRPOS_GRCh37,Canonical_SPDI,Gene_name,Functional_consequence,Validation_flags,ALFA_freq,ALFA_AC,ALSPAC_freq,ALSPAC_AC,TOMMO_freq,TOMMO_AC
0,rs2233648,snv,C>G,4:73981419,4:74847136,NC_000004.12:73981418:C:G,PF4,"coding_sequence_variant,synonymous_variant","by-frequency,by-alfa,by-cluster",0.005184,244.0,0.0,0.0,0.000039,3.0
1,rs367951530,snv,G>A;G>T,4:73981468,4:74847185,"NC_000004.12:73981467:G:A,NC_000004.12:7398146...",PF4,"coding_sequence_variant,missense_variant","by-frequency,by-alfa,by-cluster",0.000058,3.0,NaN,NaN,NaN,NaN
2,rs764474600,snv,C>G;C>T,4:73981460,4:74847177,"NC_000004.12:73981459:C:G,NC_000004.12:7398145...",PF4,"coding_sequence_variant,missense_variant","by-frequency,by-alfa,by-cluster",0.000000,0.0,NaN,NaN,0.000013,1.0
3,rs765704070,snv,C>G;C>T,4:73981445,4:74847162,"NC_000004.12:73981444:C:G,NC_000004.12:7398144...",PF4,"coding_sequence_variant,missense_variant","by-frequency,by-alfa,by-cluster",0.000000,0.0,NaN,NaN,NaN,NaN
4,rs982761496,snv,C>A,4:73981500,4:74847217,NC_000004.12:73981499:C:A,PF4,"coding_sequence_variant,missense_variant","by-frequency,by-alfa",0.000000,0.0,NaN,NaN,NaN,NaN


In [6]:
freq_cols = ["ALFA_freq", "ALSPAC_freq", "TOMMO_freq"]

df[freq_cols] = df[freq_cols].apply(pd.to_numeric, errors="coerce")

df["MAX_freq"] = df[freq_cols].max(axis=1)

df["Priority"] = df["MAX_freq"].apply(
    lambda x: "Primary" if pd.notna(x) and x > 0.001
    else "Secondary" if pd.notna(x) and 0.0001 < x <= 0.001
    else "Below threshold"
)

df.head()

,rsID,Variant_type,Alleles_REF_ALT,CHRPOS_GRCh38,CHRPOS_GRCh37,Canonical_SPDI,Gene_name,Functional_consequence,Validation_flags,ALFA_freq,ALFA_AC,ALSPAC_freq,ALSPAC_AC,TOMMO_freq,TOMMO_AC,MAX_freq,Priority
0,rs2233648,snv,C>G,4:73981419,4:74847136,NC_000004.12:73981418:C:G,PF4,"coding_sequence_variant,synonymous_variant","by-frequency,by-alfa,by-cluster",0.005184,244.0,0.0,0.0,0.000039,3.0,0.005184,Primary
1,rs367951530,snv,G>A;G>T,4:73981468,4:74847185,"NC_000004.12:73981467:G:A,NC_000004.12:7398146...",PF4,"coding_sequence_variant,missense_variant","by-frequency,by-alfa,by-cluster",0.000058,3.0,NaN,NaN,NaN,NaN,0.000058,Below threshold
2,rs764474600,snv,C>G;C>T,4:73981460,4:74847177,"NC_000004.12:73981459:C:G,NC_000004.12:7398145...",PF4,"coding_sequence_variant,missense_variant","by-frequency,by-alfa,by-cluster",0.000000,0.0,NaN,NaN,0.000013,1.0,0.000013,Below threshold
3,rs765704070,snv,C>G;C>T,4:73981445,4:74847162,"NC_000004.12:73981444:C:G,NC_000004.12:7398144...",PF4,"coding_sequence_variant,missense_variant","by-frequency,by-alfa,by-cluster",0.000000,0.0,NaN,NaN,NaN,NaN,0.000000,Below threshold
4,rs982761496,snv,C>A,4:73981500,4:74847217,NC_000004.12:73981499:C:A,PF4,"coding_sequence_variant,missense_variant","by-frequency,by-alfa",0.000000,0.0,NaN,NaN,NaN,NaN,0.000000,Below threshold


In [7]:
with open(region_path, "r") as f:
    pf4_region = json.load(f)

pf4_chromosome = str(pf4_region["chromosome"])
gene_start = pf4_region["gene_start"]
gene_end = pf4_region["gene_end"]
region_start = pf4_region["region_start"]
region_end = pf4_region["region_end"]


def extract_position_grch38(chrpos):
    if pd.isna(chrpos):
        return None

    parts = str(chrpos).split(":")

    if len(parts) != 2:
        return None

    chromosome, position = parts

    if chromosome != pf4_chromosome:
        return None

    try:
        return int(position)
    except ValueError:
        return None


def annotate_location_relative_to_pf4(position):
    if pd.isna(position):
        return "unknown"

    if gene_start <= position <= gene_end:
        return "inside_PF4"

    if region_start <= position <= region_end:
        return "near_PF4"

    return "outside_PF4_region"


df["Location_relative_to_PF4"] = df["CHRPOS_GRCh38"].apply(
    lambda chrpos: annotate_location_relative_to_pf4(
        extract_position_grch38(chrpos)
    )
)

df.head()

,rsID,Variant_type,Alleles_REF_ALT,CHRPOS_GRCh38,CHRPOS_GRCh37,Canonical_SPDI,Gene_name,Functional_consequence,Validation_flags,ALFA_freq,ALFA_AC,ALSPAC_freq,ALSPAC_AC,TOMMO_freq,TOMMO_AC,MAX_freq,Priority,Location_relative_to_PF4
0,rs2233648,snv,C>G,4:73981419,4:74847136,NC_000004.12:73981418:C:G,PF4,"coding_sequence_variant,synonymous_variant","by-frequency,by-alfa,by-cluster",0.005184,244.0,0.0,0.0,0.000039,3.0,0.005184,Primary,inside_PF4
1,rs367951530,snv,G>A;G>T,4:73981468,4:74847185,"NC_000004.12:73981467:G:A,NC_000004.12:7398146...",PF4,"coding_sequence_variant,missense_variant","by-frequency,by-alfa,by-cluster",0.000058,3.0,NaN,NaN,NaN,NaN,0.000058,Below threshold,inside_PF4
2,rs764474600,snv,C>G;C>T,4:73981460,4:74847177,"NC_000004.12:73981459:C:G,NC_000004.12:7398145...",PF4,"coding_sequence_variant,missense_variant","by-frequency,by-alfa,by-cluster",0.000000,0.0,NaN,NaN,0.000013,1.0,0.000013,Below threshold,inside_PF4
3,rs765704070,snv,C>G;C>T,4:73981445,4:74847162,"NC_000004.12:73981444:C:G,NC_000004.12:7398144...",PF4,"coding_sequence_variant,missense_variant","by-frequency,by-alfa,by-cluster",0.000000,0.0,NaN,NaN,NaN,NaN,0.000000,Below threshold,inside_PF4
4,rs982761496,snv,C>A,4:73981500,4:74847217,NC_000004.12:73981499:C:A,PF4,"coding_sequence_variant,missense_variant","by-frequency,by-alfa",0.000000,0.0,NaN,NaN,NaN,NaN,0.000000,Below threshold,inside_PF4


In [8]:
# Drop fully identical duplicate rows
df = df.drop_duplicates()

In [9]:
df.to_csv(out_csv, index=False)

In [10]:
summary = {
    "output_file": str(out_csv),
    "total_rows": int(len(df)),
    "unique_rsIDs": int(df["rsID"].nunique(dropna=True)),
    "priority_counts": df["Priority"].value_counts(dropna=False).to_dict(),
    "location_counts": df["Location_relative_to_PF4"].value_counts(dropna=False).to_dict(),
}

summary

{'output_file': '../data/final/pf4_variants.csv',
 'total_rows': 1976,
 'unique_rsIDs': 1976,
 'priority_counts': {'Below threshold': 1800, 'Secondary': 128, 'Primary': 48},
 'location_counts': {'near_PF4': 1205, 'inside_PF4': 771}}